# Linear Regression: Complete Guide

## Learning Objectives
By the end of this notebook, you will:
- Understand the mathematical foundations of linear regression
- Implement linear regression from scratch using gradient descent
- Master regularization techniques (Ridge, Lasso, Elastic Net)
- Evaluate regression models with appropriate metrics
- Handle polynomial features and feature scaling
- Diagnose and fix common regression problems

## Prerequisites
- NumPy fundamentals
- Basic calculus (derivatives)
- Statistics basics (mean, variance, correlation)

---
## Part 1: Simple Linear Regression Theory

### The Model
Linear regression finds the best-fitting line through data points:

$$\hat{y} = \beta_0 + \beta_1 x$$

Where:
- $\hat{y}$ is the predicted value
- $\beta_0$ is the intercept (y-value when x=0)
- $\beta_1$ is the slope (change in y per unit change in x)
- $x$ is the input feature

### The Goal
Find $\beta_0$ and $\beta_1$ that minimize the **Mean Squared Error (MSE)**:

$$MSE = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded successfully!")

In [ ]:
# Generate simple data for visualization
np.random.seed(42)
X_simple = np.linspace(0, 10, 50)
y_simple = 2.5 * X_simple + 3 + np.random.normal(0, 2, 50)

# Visualize the data
plt.figure(figsize=(10, 6))
plt.scatter(X_simple, y_simple, alpha=0.7, s=60, label='Data points')
plt.xlabel('X (Feature)')
plt.ylabel('y (Target)')
plt.title('Simple Linear Regression: Finding the Best Fit Line')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"True relationship: y = 2.5x + 3 + noise")

---
## Part 2: Linear Regression from Scratch

### Closed-Form Solution (Normal Equation)
For the model $y = X\beta$, the optimal solution is:

$$\beta = (X^T X)^{-1} X^T y$$

### Gradient Descent Approach
Update weights iteratively:
$$\beta_{t+1} = \beta_t - \eta \cdot \nabla MSE(\beta_t)$$

Where the gradient is:
$$\nabla MSE = -\frac{2}{n} X^T (y - X\beta)$$

In [ ]:
class LinearRegressionScratch:
    """Linear Regression implemented from scratch."""
    
    def __init__(self, method='normal', learning_rate=0.01, n_iterations=1000):
        self.method = method  # 'normal' or 'gradient'
        self.lr = learning_rate
        self.n_iter = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        if self.method == 'normal':
            # Add bias column
            X_b = np.c_[np.ones(n_samples), X]
            # Normal equation: beta = (X^T X)^-1 X^T y
            theta = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
            self.bias = theta[0]
            self.weights = theta[1:]
            
        else:  # Gradient descent
            self.weights = np.zeros(n_features)
            self.bias = 0
            
            for i in range(self.n_iter):
                # Predictions
                y_pred = X @ self.weights + self.bias
                
                # Gradients
                dw = -(2/n_samples) * X.T @ (y - y_pred)
                db = -(2/n_samples) * np.sum(y - y_pred)
                
                # Update weights
                self.weights -= self.lr * dw
                self.bias -= self.lr * db
                
                # Track loss
                loss = np.mean((y - y_pred) ** 2)
                self.loss_history.append(loss)
                
        return self
    
    def predict(self, X):
        return X @ self.weights + self.bias
    
    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)
    
print("LinearRegressionScratch class defined!")

In [ ]:
# Test our implementation
X = X_simple.reshape(-1, 1)
y = y_simple

# Normal equation method
model_normal = LinearRegressionScratch(method='normal')
model_normal.fit(X, y)

# Gradient descent method
X_scaled = (X - X.mean()) / X.std()  # Scale for gradient descent
model_gd = LinearRegressionScratch(method='gradient', learning_rate=0.1, n_iterations=100)
model_gd.fit(X_scaled, y)

print("Normal Equation Results:")
print(f"  Intercept: {model_normal.bias:.4f}")
print(f"  Coefficient: {model_normal.weights[0]:.4f}")
print(f"  R² Score: {model_normal.score(X, y):.4f}")

print("\nGradient Descent Results (scaled X):")
print(f"  Intercept: {model_gd.bias:.4f}")
print(f"  R² Score: {model_gd.score(X_scaled, y):.4f}")

In [ ]:
# Visualize gradient descent convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Fitted line
axes[0].scatter(X, y, alpha=0.7, label='Data')
X_line = np.linspace(0, 10, 100).reshape(-1, 1)
axes[0].plot(X_line, model_normal.predict(X_line), 'r-', lw=2, label='Fitted line')
axes[0].set_xlabel('X')
axes[0].set_ylabel('y')
axes[0].set_title('Linear Regression Fit')
axes[0].legend()

# Plot 2: Loss convergence
axes[1].plot(model_gd.loss_history)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('MSE Loss')
axes[1].set_title('Gradient Descent Convergence')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

---
## Part 3: Using Scikit-Learn

Scikit-learn provides optimized implementations with additional features.

In [ ]:
# Compare with sklearn
from sklearn.linear_model import LinearRegression

sklearn_model = LinearRegression()
sklearn_model.fit(X, y)

print("Comparison: Our Implementation vs Scikit-Learn")
print("=" * 50)
print(f"{'Metric':<20} {'Scratch':<15} {'Sklearn':<15}")
print("-" * 50)
print(f"{'Intercept':<20} {model_normal.bias:<15.4f} {sklearn_model.intercept_:<15.4f}")
print(f"{'Coefficient':<20} {model_normal.weights[0]:<15.4f} {sklearn_model.coef_[0]:<15.4f}")
print(f"{'R² Score':<20} {model_normal.score(X, y):<15.4f} {sklearn_model.score(X, y):<15.4f}")

---
## Part 4: Multiple Linear Regression

With multiple features, the model becomes:
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_p x_p$$

In [ ]:
# Load California Housing dataset
housing = fetch_california_housing()
X_housing = pd.DataFrame(housing.data, columns=housing.feature_names)
y_housing = housing.target

print("California Housing Dataset:")
print(f"  Samples: {X_housing.shape[0]}")
print(f"  Features: {X_housing.shape[1]}")
print(f"  Target: Median house value (in $100,000s)")
print(f"\nFeatures: {list(X_housing.columns)}")
X_housing.head()

In [ ]:
# Quick EDA
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(X_housing.columns):
    axes[idx].hist(X_housing[col], bins=30, edgecolor='black', alpha=0.7)
    axes[idx].set_title(col)
    axes[idx].set_xlabel('')
    
plt.suptitle('Feature Distributions - California Housing', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Feature scaling (important for regularized models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = model.predict(X_train_scaled)
y_pred_test = model.predict(X_test_scaled)

print("Multiple Linear Regression Results:")
print(f"  Training R²: {r2_score(y_train, y_pred_train):.4f}")
print(f"  Test R²: {r2_score(y_test, y_pred_test):.4f}")
print(f"  Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.4f}")
print(f"  Test MAE: {mean_absolute_error(y_test, y_pred_test):.4f}")

In [ ]:
# Feature importance (coefficients)
coef_df = pd.DataFrame({
    'Feature': X_housing.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if c > 0 else 'red' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient Value')
plt.title('Feature Importance (Standardized Coefficients)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

print("\nCoefficient Interpretation:")
print("  Green = Positive effect on house price")
print("  Red = Negative effect on house price")

---
## Part 5: Regularization (Ridge, Lasso, Elastic Net)

Regularization prevents overfitting by adding a penalty term to the loss function.

### Ridge Regression (L2)
$$Loss = MSE + \alpha \sum_{j=1}^{p} \beta_j^2$$

### Lasso Regression (L1)
$$Loss = MSE + \alpha \sum_{j=1}^{p} |\beta_j|$$

### Elastic Net (L1 + L2)
$$Loss = MSE + \alpha_1 \sum |\beta_j| + \alpha_2 \sum \beta_j^2$$

In [ ]:
# Compare regularization methods
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]

results = {'alpha': alphas}
for model_name, ModelClass in [('Ridge', Ridge), ('Lasso', Lasso)]:
    train_scores = []
    test_scores = []
    
    for alpha in alphas:
        model = ModelClass(alpha=alpha, max_iter=10000)
        model.fit(X_train_scaled, y_train)
        train_scores.append(model.score(X_train_scaled, y_train))
        test_scores.append(model.score(X_test_scaled, y_test))
    
    results[f'{model_name}_train'] = train_scores
    results[f'{model_name}_test'] = test_scores

results_df = pd.DataFrame(results)
print("Regularization Comparison:")
display(results_df.round(4))

In [ ]:
# Visualize regularization effect
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge vs Lasso scores
for i, model_name in enumerate(['Ridge', 'Lasso']):
    axes[i].semilogx(alphas, results_df[f'{model_name}_train'], 'b-o', label='Train')
    axes[i].semilogx(alphas, results_df[f'{model_name}_test'], 'r-s', label='Test')
    axes[i].set_xlabel('Alpha (Regularization Strength)')
    axes[i].set_ylabel('R² Score')
    axes[i].set_title(f'{model_name} Regression')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Lasso for feature selection
lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

# Show which features are selected (non-zero coefficients)
lasso_coef = pd.DataFrame({
    'Feature': X_housing.columns,
    'Coefficient': lasso.coef_,
    'Selected': lasso.coef_ != 0
}).sort_values('Coefficient', key=abs, ascending=False)

print("Lasso Feature Selection (alpha=0.1):")
print(f"  Features selected: {lasso_coef['Selected'].sum()} / {len(lasso_coef)}")
display(lasso_coef)

---
## Part 6: Polynomial Regression

For non-linear relationships, we can add polynomial features:
$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3 + ...$$

In [ ]:
# Create non-linear data
np.random.seed(42)
X_poly = np.linspace(-3, 3, 100)
y_poly = 0.5 * X_poly**3 - 2 * X_poly**2 + X_poly + 3 + np.random.normal(0, 2, 100)

# Try different polynomial degrees
degrees = [1, 2, 3, 5, 10]
X_poly_2d = X_poly.reshape(-1, 1)

fig, axes = plt.subplots(1, len(degrees), figsize=(20, 4))

for idx, degree in enumerate(degrees):
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_features = poly.fit_transform(X_poly_2d)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_poly_features, y_poly)
    
    # Predict
    X_line = np.linspace(-3, 3, 200).reshape(-1, 1)
    X_line_poly = poly.transform(X_line)
    y_line = model.predict(X_line_poly)
    
    # Plot
    axes[idx].scatter(X_poly, y_poly, alpha=0.5, s=20)
    axes[idx].plot(X_line, y_line, 'r-', lw=2)
    axes[idx].set_title(f'Degree {degree}\nR² = {model.score(X_poly_features, y_poly):.3f}')
    axes[idx].set_ylim(-20, 30)

plt.suptitle('Polynomial Regression: Effect of Degree', fontsize=14)
plt.tight_layout()
plt.show()

print("⚠️ Warning: High-degree polynomials can overfit!")

---
## Part 7: Model Evaluation Metrics

### Regression Metrics

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **MSE** | $\frac{1}{n}\sum(y-\hat{y})^2$ | Average squared error |
| **RMSE** | $\sqrt{MSE}$ | Error in original units |
| **MAE** | $\frac{1}{n}\sum|y-\hat{y}|$ | Average absolute error |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Variance explained (0-1) |
| **Adjusted R²** | Penalizes extra features | Better for multiple regression |

In [ ]:
def evaluate_regression(y_true, y_pred, n_features=None):
    """Comprehensive regression evaluation."""
    n = len(y_true)
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Adjusted R²
    if n_features:
        adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)
    else:
        adj_r2 = None
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'R²': r2,
        'Adjusted R²': adj_r2
    }

# Evaluate our housing model
metrics = evaluate_regression(y_test, y_pred_test, n_features=X_housing.shape[1])
print("Model Evaluation Metrics:")
for metric, value in metrics.items():
    if value is not None:
        print(f"  {metric}: {value:.4f}")

In [ ]:
# Residual analysis
residuals = y_test - y_pred_test

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1. Residuals vs Predicted
axes[0, 0].scatter(y_pred_test, residuals, alpha=0.3)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Predicted')

# 2. Residual distribution
axes[0, 1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].axvline(x=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Residual Value')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Residual Distribution')

# 3. Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot (Normality Check)')

# 4. Actual vs Predicted
axes[1, 1].scatter(y_test, y_pred_test, alpha=0.3)
axes[1, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1, 1].set_xlabel('Actual Values')
axes[1, 1].set_ylabel('Predicted Values')
axes[1, 1].set_title('Actual vs Predicted')

plt.suptitle('Residual Analysis', fontsize=14)
plt.tight_layout()
plt.show()

---
## Part 8: Cross-Validation for Regression

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

# 5-fold cross-validation
models = {
    'Linear': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1, max_iter=10000),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)
}

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    cv_results[name] = {
        'Mean R²': scores.mean(),
        'Std': scores.std(),
        'Min': scores.min(),
        'Max': scores.max()
    }

cv_df = pd.DataFrame(cv_results).T
print("5-Fold Cross-Validation Results:")
display(cv_df.round(4))

---
## Part 9: Practice Exercises

### Exercise 1: Implement Gradient Descent with Momentum
Add momentum to speed up gradient descent convergence.

### Exercise 2: Compare Regularization Strengths
Find the optimal alpha for Ridge and Lasso using cross-validation.

### Exercise 3: Build a Complete Pipeline
Create a preprocessing + regression pipeline for the housing data.

In [ ]:
# Exercise 2 Solution: Finding Optimal Alpha
from sklearn.linear_model import RidgeCV, LassoCV

# RidgeCV with built-in cross-validation
ridge_cv = RidgeCV(alphas=np.logspace(-4, 4, 50), cv=5)
ridge_cv.fit(X_train_scaled, y_train)

# LassoCV with built-in cross-validation
lasso_cv = LassoCV(alphas=np.logspace(-4, 2, 50), cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

print("Optimal Regularization Strength:")
print(f"  Ridge optimal alpha: {ridge_cv.alpha_:.4f}")
print(f"  Ridge test R²: {ridge_cv.score(X_test_scaled, y_test):.4f}")
print(f"\n  Lasso optimal alpha: {lasso_cv.alpha_:.4f}")
print(f"  Lasso test R²: {lasso_cv.score(X_test_scaled, y_test):.4f}")

In [ ]:
# Exercise 3 Solution: Complete Pipeline
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=ridge_cv.alpha_))
])

# Fit and evaluate
pipeline.fit(X_train, y_train)
print("Pipeline Results:")
print(f"  Training R²: {pipeline.score(X_train, y_train):.4f}")
print(f"  Test R²: {pipeline.score(X_test, y_test):.4f}")

# Cross-validation on full pipeline
cv_scores = cross_val_score(pipeline, X_housing, y_housing, cv=5, scoring='r2')
print(f"\n5-Fold CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

---
## Summary

### Key Takeaways

| Concept | Description |
|---------|-------------|
| **Linear Regression** | Models linear relationship between X and y |
| **Normal Equation** | Closed-form solution: $\beta = (X^TX)^{-1}X^Ty$ |
| **Gradient Descent** | Iterative optimization for large datasets |
| **Ridge (L2)** | Shrinks coefficients, keeps all features |
| **Lasso (L1)** | Feature selection, zeros out some coefficients |
| **Elastic Net** | Combines L1 and L2 penalties |
| **Polynomial Features** | Captures non-linear relationships |

### When to Use What

| Situation | Recommendation |
|-----------|---------------|
| Few features, no multicollinearity | Standard Linear Regression |
| Many correlated features | Ridge Regression |
| Need feature selection | Lasso Regression |
| Non-linear patterns | Polynomial Features + Regularization |
| Production system | Always use pipelines + cross-validation |

In [ ]:
# Final verification
print("=" * 60)
print("Linear Regression Notebook Complete!")
print("=" * 60)
print(f"\nModels covered:")
print("  ✅ Simple Linear Regression")
print("  ✅ Multiple Linear Regression")
print("  ✅ Ridge Regression (L2)")
print("  ✅ Lasso Regression (L1)")
print("  ✅ Elastic Net")
print("  ✅ Polynomial Regression")
print("\nKey skills:")
print("  ✅ From-scratch implementation")
print("  ✅ Scikit-learn usage")
print("  ✅ Model evaluation & diagnostics")
print("  ✅ Cross-validation")
print("  ✅ Hyperparameter tuning")